# Basics
This notebook introduces ``Data`` placeholders and callable ``Transformation`` objects.

## Defining transformations
A ``Data`` object is a placeholder for values supplied when a transformation is called.
A transformation can use one or more data placeholders and can be composed with other transformations.

In [4]:
import numpy as np

from py_online_forecast.core import *
from py_online_forecast.features import One, LowPass

A simple ``Transformation`` is ``One``, which creates an array of ones with the same length as a reference input.

In [5]:
data_source = Data()
constant = One(data_source)

Once data becomes available, call the transformation with a dictionary that maps the ``Data`` placeholder to its value.

In [6]:
# Create a data value
data_value = np.array([1, 2, 3, 4, 5])

# Apply the transformation
constant({data_source: data_value})

array([1., 1., 1., 1., 1.])

Transformations also support common operators. These operators create new transformations which can be evaluated later.

In [7]:
# Define a transformation by adding 1 to the data source
data_plus_one = data_source + 1

# Create another transformation by indexing the new transformation
first_three = data_plus_one[:3]

# Apply the new transformation
first_three({data_source: data_value})

array([2, 3, 4])

Transformations can also use ``DEFAULT_SOURCE`` when there is only one unnamed input:

In [8]:
# Create a transformation using DEFAULT_SOURCE
data_plus_one = DEFAULT_SOURCE + 1
first_three = data_plus_one[:3]

# Evaluate without explicitly naming the source
first_three(data_value)

array([2, 3, 4])

## Online updates
Some transformations use state to process data incrementally. ``LowPass`` is one example: its state allows a later batch to continue the filtering started by an earlier batch.

In [12]:
data = np.random.rand(6, 1)

# Create a low-pass filter transformation
low_pass = LowPass(DEFAULT_SOURCE, alpha=0.1)

# Apply the low-pass filter to the data
first_three = low_pass(data[:3])
last_three = low_pass(data[3:6])
all_data = low_pass(data)

# Check where they are the same
np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [False],
       [False],
       [False]])

To continue a calculation explicitly, request the states produced by one call and pass them to the next call.

In [13]:
first_three, states = low_pass(data[:3], return_states=True)
states

defaultdict(<function py_online_forecast.core.Transformation.__call__.<locals>.<lambda>()>,
            {<py_online_forecast.features.LowPass at 0x7b51b6c93aa0>: array([0.27422147])})

The result is a tuple containing the output and a state mapping. The mapping uses transformations as keys and stores the state produced by each stateful transformation.

Pass the mapping back with ``states=`` to resume the calculation:

In [14]:
last_three = low_pass(data[3:6], states = states)

np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True]])

Alternatively, ask the transformation to retain its states internally by passing ``track_states=True``.

In [15]:
first_three = low_pass(data[:3], track_states = True)
last_three = low_pass(data[3:6], track_states = True)
np.isclose(np.concatenate([first_three, last_three]), all_data)

array([[ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True]])

## Updating parameters
Transformation parameters can be a mix of Transformations and fixed values. Fixed parameters can be modified as ordinary python attributes, which for example is the case with the LowPass filter parameter.

In [16]:
low_pass.alpha = 0.2

print("With alpha = 0.1:")
print(all_data)

print("With alpha = 0.2:")
print(low_pass(data))

With alpha = 0.1:
[[0.66587526]
 [0.88938805]
 [0.27422147]
 [0.28520696]
 [0.1658257 ]
 [0.21375418]]
With alpha = 0.2:
[[0.66587526]
 [0.8645533 ]
 [0.33760637]
 [0.29666333]
 [0.18138156]
 [0.21153997]]


It is also possible to use the purpose built Param transformation, which simply returns a fixed value, but can be overwritten by passing it as an input.